# NAMD-FEP — notebook 2 · prepare & sanitise the inputs

Turn the complex from notebook 01 into the four things `rbfe` needs:

```
inputs/protein.pdb           the protein (hydrogens optional — psfgen adds them)
inputs/ref.{pdb,rtf,prm}     the reference ligand, all-atom, parameterised
inputs/mut.{pdb,rtf,prm}     the mutant ligand, all-atom, parameterised
system.ini                   the one config the whole pipeline reads
```

Every step **shows you what it made** — the 3D structure, the 2D depiction, the
perceived SMILES and charge — because the tools *perceive* chemistry, they do not
*know* it. The human checks between the cells are the point of this notebook.

The worked example throughout is 6I5I (CLK1, ligand H3E): a **desmethylation**
N–CH₃ → N–H, so the mutant is derived from the crystal ligand by deleting the
methyl group (`--drop`). That keeps every shared atom in its crystallographic pose.

> Needs: `obabel` + `acpype` for the parameterisation cells (notebook 00 installs
> them; on Binder they come from `notebooks/environment.yml`). Without them the
> visualisation cells still run and the parameterisation cell says what to install.

## 2.1 Setup and choices

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():
    REPO = REPO.parent
os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

WORK = REPO / "_tutorial"
SYS  = WORK / "6i5i"                      # the system directory (one system = one ini)
(SYS / "inputs").mkdir(parents=True, exist_ok=True)

saved = {}
if (WORK / "choices.json").exists():
    saved = json.loads((WORK / "choices.json").read_text())

COMPLEX_PDB = Path(saved.get("complex_pdb", WORK / "6I5I.pdb"))
CHAIN   = saved.get("chain", "A")
LIG_RES = saved.get("ligand_resname", "H3E")

print("complex :", COMPLEX_PDB, "(exists)" if COMPLEX_PDB.exists() else "(MISSING -- run notebook 01)")
print("chain   :", repr(CHAIN))
print("ligand  :", LIG_RES)
print("system  :", SYS)

## 2.2 Sanitise the protein — `rbfe-prep strip`

Keeps the chosen chain's `ATOM` records, resolves altlocs (blank/major conformer),
and drops every `HETATM` — water, ions, crystallisation additives *and the ligand*
(the ligand is re-added on its own in 2.3, hydrogens completed). Protein hydrogens
may be absent: psfgen adds them at build time.

In [ ]:
protein = SYS / "inputs" / "protein.pdb"

cmd = ["rbfe-prep", "strip", str(COMPLEX_PDB), "-o", str(protein)]
if CHAIN:
    cmd += ["--chain", CHAIN]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout, r.stderr)
r.check_returncode()

import nglview as nv
view = nv.show_structure_file(str(protein))
view.layout.width = "100%"
view.layout.height = "600px"  # fixed canvas: on JupyterLab the stage is sized before its container exists, which leaves a blank widget
view.clear_representations()
view.add_representation("cartoon", color="sstruc")
view.center()
print("the sanitised protein — no ligand, no water, one conformer")
display(view)
view.handle_resize()  # force the stage to recalc its size once visible


## 2.3 Extract and sanitise the two ligands — `rbfe-prep ligand`

Each ligand is one HETATM residue pulled out of the complex, converted to a
**hydrogen-complete SDF in the crystal frame** (obabel adds the hydrogens and
perceives the bond orders).

The two ligands of an RBFE pair differ by a small edit. Here the **mutant** is the
crystal ligand as-is, and the **reference** is derived by `--drop`-ing the methyl
group (atoms `C12 H7 H8 H9` in 6I5I's H3E) — the mutation runs N–CH₃ → N–H.
Deriving the smaller ligand this way guarantees the shared atoms keep the crystal
coordinates.

> **Read what this prints.** The formula, SMILES and atom count are *perceived* by
> obabel, not known. If the SMILES is not your molecule, re-run with
> `--smiles "..."` — the crystal frame is kept and only the bond orders are taken
> from the SMILES.

In [ ]:
DROP = ["C12", "H7", "H8", "H9"]     # <-- the methyl removed to make the reference
SMILES_OVERRIDE = None               # <-- or e.g. "Cc1..." if perception is wrong

import shutil

if shutil.which("obabel") is None:
    print("obabel not found -- skipping ligand extraction.\n"
          "  conda install -c conda-forge openbabel   (it is in the Binder env)\n"
          "Everything downstream of this cell in this notebook needs the SDFs it writes.")
    HAVE_SDF = False
else:
    HAVE_SDF = True
    for name, extra in (("mut", []), ("ref", ["--drop", *DROP])):
        out = SYS / "inputs" / f"{name}.sdf"
        cmd = ["rbfe-prep", "ligand", str(COMPLEX_PDB), LIG_RES, "-o", str(out)]
        if CHAIN:
            cmd += ["--chain", CHAIN]
        cmd += (["--smiles", SMILES_OVERRIDE] if SMILES_OVERRIDE else extra)
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout, r.stderr)
        r.check_returncode()
        print()

# true also when the SDFs are left over from a previous run
HAVE_SDF = HAVE_SDF or all((SYS / "inputs" / f"{n}.sdf").exists()
                           for n in ("ref", "mut"))

## 2.4 Look at the pair

Left: 2D depictions (RDKit). Right: the two ligands overlaid in 3D — they should sit
on top of each other except where the mutation is. **If the 2D graphs do not match
the 3D structures, stop and fix the SMILES before going on.**

In [ ]:
have_sdf = globals().get("HAVE_SDF", all((SYS / "inputs" / f"{n}.sdf").exists()
                                           for n in ("ref", "mut")))
if not have_sdf:
    print("no ligand SDFs yet -- the extraction cell above was skipped")
else:
    try:
        from rdkit import Chem
        from rdkit.Chem import Draw
        mols, labels = [], []
        for name in ("ref", "mut"):
            suppl = Chem.SDMolSupplier(str(SYS / "inputs" / f"{name}.sdf"), removeHs=False)
            m = next(iter(suppl))
            mols.append(m)
            labels.append(f"{name}: {Chem.MolToSmiles(Chem.RemoveHs(m))}")
        img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(360, 300), legends=labels)
        display(img)
        for lab in labels:
            print(lab)
    except ImportError:
        print("rdkit not installed -- skipping the 2D depiction (3D view below still works)")

In [ ]:
import nglview as nv

if not globals().get("have_sdf", False):
    print("no ligand SDFs yet -- the extraction cell above was skipped")
else:
    view = nv.NGLWidget()
    view.layout.width = "100%"
    view.layout.height = "600px"  # fixed canvas: blank on JupyterLab without it
    c1 = view.add_component(str(SYS / "inputs" / "mut.sdf"))
    c2 = view.add_component(str(SYS / "inputs" / "ref.sdf"))
    view.clear_representations()
    c1.add_representation("licorice", color="green")
    c2.add_representation("licorice", color="magenta")
    view.center()
    print("mutant (green) overlaid on reference (magenta) -- same crystal frame")
    display(view)
    view.handle_resize()  # force the stage to recalc its size once visible


## 2.5 Parameterise — `rbfe-prep params`

Runs **acpype** (AM1-BCC charges, GAFF2) and writes `ref.{rtf,prm,pdb,mol2}` and
`mut.{rtf,prm,pdb,mol2}`. This is the slow cell (a few minutes per ligand).

Two things to check in the output, every time:

1. **`RESI total charge`** must be within ~0.02 of an integer — a non-integer means a
   truncated or hand-edited `.rtf`, and `rbfe hybrid` will refuse it. Pass `--net`
   with the intended formal charge when it is not 0.
2. the atom names in the `.pdb` are the names `[mutation] vanish/appear` will refer
   to in notebook 03.

In [ ]:
import shutil

NET_CHARGE = 0                          # <-- formal net charge of the ligands

if shutil.which("acpype") is None or shutil.which("obabel") is None:
    print("acpype/obabel not found. Install with:\n"
          "  pip install acpype\n"
          "  conda install -c conda-forge openbabel\n"
          "(on Binder they are in notebooks/environment.yml)")
else:
    for name in ("ref", "mut"):
        r = subprocess.run(["rbfe-prep", "params",
                            str(SYS / "inputs" / f"{name}.sdf"),
                            "--name", name, "-o", str(SYS / "inputs"),
                            "--net", str(NET_CHARGE)],
                           capture_output=True, text=True)
        # print only the verdict lines; acpype is chatty
        keep = [l for l in (r.stdout + r.stderr).splitlines()
                if any(k in l for k in ("wrote", "RESI", "charge", "atoms", "ERROR", "error"))]
        print("\n".join(keep) or r.stdout[-2000:])
        r.check_returncode()
        print()
    print("parameterised:", sorted(p.name for p in (SYS / 'inputs').iterdir()))

## 2.6 Scaffold the system — `rbfe-prep scaffold`

Writes `system.ini`. The `[mutation]` block is **deliberately blank** — choosing the
strategy is a chemistry decision, and `rbfe` refuses to guess. Fill it in next.

In [ ]:
r = subprocess.run(["rbfe-prep", "scaffold", str(SYS),
                    "--chain", CHAIN or "A", "--resname", "UNL",
                    "--description", f"{COMPLEX_PDB.stem} {LIG_RES} tutorial",
                    "--force"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
r.check_returncode()

## 2.7 Declare the mutation

Three strategies:

| strategy | when | `vanish` / `appear` |
|---|---|---|
| `element_swap` | one atom becomes a different element (I → Br) | atom names, same in both |
| `atom_addition` | the mutant has an atom the reference lacks (N–CH₃ → N–H) | vanish = the leaving group |
| `mcs` | anything else — the mapping is **derived** by RDKit | omit both |

For the tutorial pair (N–CH₃ → N–H) that is `atom_addition`, mirroring the shipped
6I5I system (`systems/6i5i/system.ini`). The cell edits `system.ini` in place — open
it and read it afterwards; it is the whole configuration of the system.

In [ ]:
import configparser, io, re

ini = SYS / "system.ini"
text = ini.read_text()

STRATEGY = "atom_addition"     # <-- element_swap / atom_addition / mcs
VANISH   = " ".join(DROP)      # atoms only the REFERENCE has (blank for mcs)
APPEAR   = "H"                 # atoms only the MUTANT has   (blank for mcs)

def set_key(text, key, value):
    pat = re.compile(rf"^({re.escape(key)}\s*=).*$", re.M)
    hits = list(pat.finditer(text))
    assert len(hits) >= 1, f"key {key!r} not found"
    return text[:hits[0].start()] + f"{key:<14}= {value}" + text[hits[0].end():]

for k, v in (("strategy", STRATEGY), ("vanish", VANISH), ("appear", APPEAR)):
    text = set_key(text, k, v)
ini.write_text(text)

# show the mutation block back
cp = configparser.ConfigParser(allow_no_value=True)
cp.optionxform = str
cp.read_string(re.sub(r"^\s+$", "", text, flags=re.M))
print(f"[mutation]\nstrategy      = {cp['mutation']['strategy']}\n"
      f"vanish        = {cp['mutation'].get('vanish') or ''}\n"
      f"appear        = {cp['mutation'].get('appear') or ''}")

## 2.8 The suitability gate — `rbfe similarity`

RBFE is a *relative* method: the number it produces is only meaningful when the two
ligands are similar enough that their configurations overlap. This scores the pair
with ECFP4 (calibrated against the repository's worked systems; threshold 0.60).
It **warns but never blocks** — read the output, not the exit code.

In [ ]:
r = subprocess.run(["rbfe", "similarity", "--json"], cwd=SYS,
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
try:
    scores = json.loads(r.stdout.strip().splitlines()[-1])
    verdict = "ok -- this pair is similar enough for RBFE" if scores["passes"] \
              else "LOW -- a relative free energy may not mean anything for this pair"
    print(f"\nECFP4 {scores['ecfp4']:.3f} vs threshold {scores['threshold']}: {verdict}")
except (ValueError, KeyError, IndexError):
    print("similarity NOT checked (rdkit missing or unreadable ligands) --\n"
          "that is a warning, not a pass. Fix it before trusting notebook 05.")

## 2.9 Where you are

✔ protein sanitised · ✔ both ligands extracted, hydrogenated, depicted · ✔
parameterised · ✔ `system.ini` written with the mutation declared · ✔ similarity
checked.

**Next: `03_hybrid_and_system.ipynb`** — build the dual-topology hybrid ligand and
the two solvated systems, and sanity-check the NAMD configs before any GPU time.